In [ ]:
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import io
import builtins
import IPython.display as ipd
from IPython.display import display as original_display, Image
def custom_display(*objs, **kwargs):
    for obj in objs:
        if hasattr(obj, 'savefig'):
            buf = io.BytesIO()
            obj.savefig(buf, format='png', bbox_inches='tight')
            buf.seek(0)
            original_display(Image(data=buf.read(), format='png'))
        else:
            original_display(obj, **kwargs)
ipd.display = custom_display
builtins.display = custom_display
def custom_show():
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    original_display(Image(data=buf.read(), format='png'))
    plt.clf()
plt.show = custom_show


In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.quantum_info import Statevector
import json, numpy as np

qc_state = QuantumCircuit(3)
qc = QuantumCircuit(3)
qc.x(0)
qc_state.x(0)
qc.x(1)
qc_state.x(1)
qc.h(2)
qc_state.h(2)

qc.measure_all()

from qiskit.visualization import circuit_drawer, plot_histogram
import matplotlib.pyplot as plt

fig = circuit_drawer(qc, output='mpl')
display(fig)
plt.close(fig)

simulator = Aer.get_backend('aer_simulator')
compiled = transpile(qc, simulator)
job = simulator.run(compiled, shots=1000)
counts = job.result().get_counts()

fig2 = plot_histogram(counts)
display(fig2)
plt.close(fig2)

# Exact (pre-measurement) state probabilities, for the interactive heatmap
sv = Statevector.from_instruction(qc_state)
probs = np.abs(sv.data) ** 2
prob_map = {format(i, '03b'): float(p) for i, p in enumerate(probs) if p > 1e-6}
print("STATE_PROBS=" + json.dumps(prob_map))
